# EDA — Analyse Exploratoire du Churn Client

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/raw/customer_churn_business_dataset.csv')
print(f"Shape: {df.shape}")
df.head()

## 1. Aperçu général

In [ ]:
print("Shape:", df.shape)
print("\nTypes:\n", df.dtypes)
print("\nValeurs manquantes:\n", df.isnull().sum())
df.describe()

## 2. Distribution de la variable cible

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['churn'].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'tomato'])
axes[0].set_title('Distribution du churn')
axes[0].set_xticklabels(['No Churn (0)', 'Churn (1)'], rotation=0)
rates = df['churn'].value_counts(normalize=True)
axes[1].pie(rates, labels=['No Churn', 'Churn'], autopct='%1.1f%%', colors=['steelblue', 'tomato'])
axes[1].set_title('Répartition churn')
plt.tight_layout()
plt.show()
print(f"Taux de churn : {df['churn'].mean():.3f} ({df['churn'].mean()*100:.1f}%)")

## 3. Distributions des variables numériques

In [ ]:
num_cols = ['age', 'tenure_months', 'monthly_logins', 'weekly_active_days',
            'avg_session_time', 'features_used', 'usage_growth_rate',
            'last_login_days_ago', 'monthly_fee', 'payment_failures',
            'support_tickets', 'csat_score', 'nps_score']

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    df[df['churn'] == 0][col].plot(kind='hist', bins=30, alpha=0.5, label='No Churn', ax=axes[i], density=True)
    df[df['churn'] == 1][col].plot(kind='hist', bins=30, alpha=0.5, label='Churn', ax=axes[i], density=True)
    axes[i].set_title(col)
    axes[i].legend(fontsize=8)
for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)
plt.tight_layout()
plt.show()

## 4. Variables catégorielles vs churn

In [ ]:
cat_cols = ['gender', 'customer_segment', 'contract_type', 'payment_method',
            'discount_applied', 'price_increase_last_3m', 'complaint_type', 'survey_response']

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    churn_rate = df.groupby(col)['churn'].mean().sort_values()
    churn_rate.plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(f'Taux de churn par {col}')
    axes[i].set_ylabel('Taux de churn')
    axes[i].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 5. Matrice de corrélation

In [ ]:
num_for_corr = ['age', 'tenure_months', 'monthly_fee', 'total_revenue',
                'payment_failures', 'support_tickets', 'csat_score',
                'nps_score', 'monthly_logins', 'churn']
corr = df[num_for_corr].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()
print("\nCorrélations avec churn (triées):\n", corr['churn'].sort_values(ascending=False))

## 6. Boxplots variables vs churn

In [ ]:
key_vars = ['tenure_months', 'monthly_fee', 'payment_failures',
            'support_tickets', 'csat_score', 'nps_score', 'monthly_logins', 'last_login_days_ago']
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(key_vars):
    df.boxplot(column=col, by='churn', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('Churn')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 7. Score d'engagement client (feature engineering)

In [ ]:
# Formule d'engagement suggérée dans le sujet
from sklearn.preprocessing import MinMaxScaler

engagement_cols = ['monthly_logins', 'weekly_active_days', 'avg_session_time', 'features_used', 'usage_growth_rate']
scaler = MinMaxScaler()
df_norm = df.copy()
df_norm[engagement_cols] = scaler.fit_transform(df[engagement_cols])
df_norm['last_login_norm'] = scaler.fit_transform(df[['last_login_days_ago']])

df['engagement_score'] = (
    0.25 * df_norm['monthly_logins'] +
    0.20 * df_norm['weekly_active_days'] +
    0.20 * df_norm['avg_session_time'] +
    0.10 * df_norm['features_used'] +
    0.10 * df_norm['usage_growth_rate'] +
    0.10 * (1 - df_norm['last_login_norm'])  # inversé : inactivité récente = mauvais signe
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['engagement_score'].hist(bins=30, ax=axes[0])
axes[0].set_title("Distribution du score d'engagement")
df.boxplot(column='engagement_score', by='churn', ax=axes[1])
axes[1].set_title("Score d'engagement vs Churn")
plt.suptitle('')
plt.tight_layout()
plt.show()
print(f"Engagement moyen (No Churn) : {df[df['churn']==0]['engagement_score'].mean():.3f}")
print(f"Engagement moyen (Churn)    : {df[df['churn']==1]['engagement_score'].mean():.3f}")

## 8. Détection des outliers (méthode IQR)

In [ ]:
def detect_outliers_iqr(df, col):
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    return n_out, lower, upper

for col in ['monthly_fee', 'total_revenue', 'payment_failures', 'support_tickets', 'avg_resolution_time']:
    n, low, up = detect_outliers_iqr(df, col)
    print(f"{col}: {n} outliers ({n/len(df)*100:.1f}%) | [{low:.1f}, {up:.1f}]")

## 9. Conclusions

### Top variables les plus liées au churn
- `contract_type` : les contrats Monthly ont un taux de churn bien plus élevé que Yearly
- `price_increase_last_3m` : une augmentation récente du prix augmente le risque
- `payment_failures` : les échecs de paiement sont un signal critique
- `csat_score` / `nps_score` : satisfaction faible = départ probable
- `complaint_type` : présence de plaintes Billing ou Technical

### Déséquilibre de classes (~10% de churn)
Dataset fortement déséquilibré. Stratégie retenue :
- `class_weight='balanced'` pour sklearn
- `scale_pos_weight` pour XGBoost
- Métriques prioritaires : **F1, Recall, ROC-AUC**

### Variables à traiter
- **Normalisation** : toutes les variables numériques (StandardScaler)
- **Encodage** : 11 variables catégorielles (OneHotEncoder)
- **Feature engineering** : score d'engagement, ratio support/tenure